In [4]:
"""Assess agent's execution trace using Claude Sonnet"""

import anthropic
import json
import pandas as pd
import re
import time
import xml.etree.ElementTree as ET
from tqdm.notebook import tqdm

from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

client = anthropic.Anthropic()

SYSTEM_PROMPT = """
Your task is to analyze the execution trace of an LLM-powered AI agent and determine why the agent failed to produce the correct answer for a given Q&A task.
You will be provided with the task, the agent's trace (JSON format), the expected answer, and the agent's answer.
The agent can write and run Python code, perform web or wikipedia searches (provided via an API), and visit webpages.
The agent can only run for at most 10 steps, afterwards it will be forced to produce an answer.

Focus on the first mistake that the agent made that led to its failure.
If it managed to fix a mistake, don't include the mistake as a failure cause.
If you cannot identify the failure cause, then you should just write UNKNOWN in the output.
Think about the issue step-by-step, then categorize the failure cause abstractly in more than three words (e.g., BAD SEARCH QUERY) and provide a brief explanation in XML format like this:
<failure_cause>
    <category>YOUR_ABSTRACT_CATEGORIZATION_OF_THE_FAILURE_CAUSE_IN_NO_MORE_THAN_THREE_WORDS</category>
    <description>YOUR_BRIEF_DESCRIPTION_OF_THE_FAILURE_CAUSE</description>
</failure_cause>
""".strip()

USER_PROMPT = """
<task>
{task}
</task>

<agent_trace>
{trace}
</agent_trace>

<expected_answer>
{answer}
</expected_answer>

<agent_answer>
{agent_output}
</agent_answer>

Now, think about the agent's failure step-by-step, then make your final conclusion in XML format like the following examples:

<failure_cause>
    <category>BAD SEARCH QUERY</category>
    <description>The agent attempted a web search with the query [QUERY], which is too specific...</description>
</failure_cause>

<failure_cause>
    <category>INCORRECT ENTITY IDENTIFICATION</category>
    <description>The agent incorrectly identified [INCORRECT_ENTITY] as ...</description>
</failure_cause>

<failure_cause>
    <category>BAD CODE GENERATION</category>
    <description>The agent generated an incorrect program...</description>
</failure_cause>
""".strip()

MODEL_ID = "claude-sonnet-4-5-20250929"

# Extract the entire execution trace from the raw input trace.json file
def extract_full_trace(trace_path, skip_system_prompt=False):
    with open(trace_path, "r") as f:
        raw_trace = json.load(f)

    # Hacky, used in eval
    def get_content(**kwargs):
        return kwargs.get("content")

    action_step = None
    for t in raw_trace[::-1]:
        if t["name"] == "ActionStep":
            action_step = t
            break

    if action_step is None:
        raise Exception("No ActionStep found")
    
    full_trace = []
    inputs = json.loads(action_step["attributes"]["output.value"])["model_input_messages"]
    output = json.loads(action_step["attributes"]["output.value"])["model_output_message"]

    for input_msg in inputs:
        content = eval(f"get_content({input_msg[input_msg.index('content='):]}")[0]["text"]
        role = re.search(r"MessageRole\.([A-Z_]+)", input_msg)
        role = role.group(1) if role is not None else ""
        full_trace.append({"role": role.lower(), "content": content})
    full_trace.append({"role": output["role"], "content": output["content"]})

    if skip_system_prompt:
        if full_trace[0]["role"] == "system":
            return full_trace[1:]

    return full_trace

In [ ]:
# Print a sample trace (skipping the first message since it's the system prompt)
for trace in extract_full_trace(f"../logs/frames_full_llamacpp_qwen3_{30}b/{5}/run_0/raw/trace.json", skip_system_prompt=True):
    print(f"===================================================\nRole: {trace['role']}\nContent:\n{trace['content']}\n")

In [ ]:
# Prepare batch for Claude
model_size = "30"
input_path = f"../data/frames/profile_results_frames_full_llamacpp_qwen3_{model_size}b_judged.csv"

df = pd.read_csv(input_path)
questions = df["question"]
answers = df["answer"]
agent_outputs = df["agent_output"]
agent_outputs_judgement = df["agent_output_eval"]

requests = []
for i in tqdm(range(len(questions))):
    # Only analyze if task was failed
    if agent_outputs_judgement[i] == "CORRECT":
        continue
    trace = extract_full_trace(f"../logs/frames_full_llamacpp_qwen3_{model_size}b/{i}/run_0/raw/trace.json", skip_system_prompt=True)
    prompt = USER_PROMPT.format(task=questions[i], trace=json.dumps(trace, indent=2), answer=answers[i], agent_output=agent_outputs[i])
    requests.append(Request(
        custom_id=f"{i}",
        params=MessageCreateParamsNonStreaming(
            model=MODEL_ID,
            max_tokens=4096,    # Max number of output tokens for Claude to generate
            system=[{
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"},
            }],
            messages=[{"role": "user", "content": prompt}],
        )
    ))

print("Batch size:", len(requests))

In [ ]:
# Submit batch
message_batch = client.messages.batches.create(requests=requests)
message_batch_id = message_batch.id
print(message_batch)

In [ ]:
# Poll until batch is done
while True:
    message_batch = client.messages.batches.retrieve(message_batch_id)
    if message_batch.processing_status == "ended":
        break
    time.sleep(10)
print("Ended!!!!!!!!!!!!!!!")

In [ ]:
# Save results
agent_output_analysis = [None] * len(questions)
for result in client.messages.batches.results(message_batch_id):
    custom_id = int(result.custom_id)
    match result.result.type:
        case "succeeded":
            res = result.result.message.content[0].text
            agent_output_analysis[custom_id] = res
        case "errored":
            print(f"Request {custom_id} failed: {result.result.error.type}")
        case "expired":
            print(f"Request expired {custom_id}")

df["agent_output_analysis"] = agent_output_analysis

output_path = f"{input_path.split('.csv')[0]}_analyzed.csv"
df.to_csv(output_path, index=False)

In [ ]:
# IGNORE THIS CELL AND BELOW

df = pd.read_csv("../data/frames/profile_results_frames_full_llamacpp_qwen3_1.7b_judged_analyzed.csv")

analyses = []
for analysis in df["agent_output_analysis"]:
    if pd.isnull(analysis):
        continue
    match = re.search(r"<category>(.*?)</category>", analysis, re.DOTALL)
    if match is None:
        print("Failed to find category in:\n" + analysis)
        continue
    category = match.group(1)

    match = re.search(r"<description>(.*?)</description>", analysis, re.DOTALL)
    if match is None:
        print("Failed to find description in:\n" + analysis)
        continue

    description = match.group(1)
    analyses.append((category, description))

In [ ]:
classifications = {
    "misinterpretations": {"keywords": ["INTERPRET", "MISUNDERST", "MISREAD"], "matches": []},
    "incompletes": {"keywords": ["INCOMPLETE"], "matches": []},
    "misidentifications": {"keywords": ["IDENTIFI"], "matches": []},
    "searches": {"keywords": ["SEARCH", "QUERY"], "matches": []},
    "reasoning": {"keywords": ["REASONING"], "matches": []},
    "extraction": {"keywords": ["EXTRACTION"], "matches": []},
    "hallucinations": {"keywords": ["HALLUCIN", "FABRICA", "UNVERIFIED"], "matches": []},
    "visits": {"keywords": ["VISIT", "EXPLOR", "RETRIEV"], "matches": []},
    "assumptions": {"keywords": ["ASSUMPTION"], "matches": []},
    "inference": {"keywords": ["INFERENCE"], "matches": []},
    "verifications": {"keywords": ["VERIF"], "matches": []},
    "logic": {"keywords": ["LOGIC"], "matches": []},
    "code": {"keywords": ["CODE"], "matches": []},
    "calculations": {"keywords": ["CALCULAT"], "matches": []},
    "loop": {"keywords": ["LOOP"], "matches": []},
    "parsing": {"keywords": ["PARSING", "PARSE"], "matches": []},
    "others": {"keywords": [], "matches": []},
}
for category_desc in analyses:
    category = category_desc[0]
    for k, v in classifications.items():
        if k == "others":
            v["matches"].append(category_desc)
            break
        keywords = v["keywords"]
        if any(kw in category for kw in keywords):
            v["matches"].append(category_desc)
            break

for k, v in classifications.items():
    print(f"{k}: {len(v['matches'])}")

for other in classifications["others"]["matches"]:
    print(other)

In [ ]:
model_size = "30"
input_path = f"../data/frames/profile_results_frames_full_llamacpp_qwen3_{model_size}b_judged.csv"
df = pd.read_csv(input_path)
questions = df["question"]
answers = df["answer"]
agent_outputs = df["agent_output"]
agent_outputs_judgement = df["agent_output_eval"].to_list()
traces = [extract_full_trace(f"../logs/frames_full_llamacpp_qwen3_{model_size}b/{i}/run_0/raw/trace.json") for i in range(len(questions))]

In [ ]:
def get_step_from_trace(trace, step_no):
    assert step_no >= 1 and isinstance(step_no, int)
    trace = [msg for msg in trace if msg["role"] in ["assistant", "tool_response"]]
    assistant_idx = [i for i, msg in enumerate(trace) if msg["role"] == "assistant"]
    # trace = [msg["content"][:len(msg["content"]) if msg["role"] == "assistant" else 100] for msg in trace]
    num_steps = len(assistant_idx)
    if step_no > num_steps:
        return None
    
    step_no -= 1
    start_idx = assistant_idx[step_no]
    end_idx = assistant_idx[step_no + 1] if step_no < num_steps - 1 else -1
    trace_range = range(start_idx, end_idx if end_idx != -1 else len(trace))
    return [trace[i]["content"] for i in trace_range]


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.metrics import classification_report, roc_auc_score

train = []
labels = []
for i, trace in enumerate(traces):
    step_msg = get_step_from_trace(trace, 4)
    if step_msg is None:
        continue
    train.append("\n".join(step_msg))
    labels.append(1 if agent_outputs_judgement[i] == "CORRECT" else 0)

# Example dataset
data = {
    "text": train,
    "label": labels,
}
df = pd.DataFrame(data)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(df["text"], df["label"], stratify=df["label"], test_size=0.2, random_state=42)

# Pipeline: TF-IDF → Logistic Regression
pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("clf", LogisticRegression(max_iter=2000, class_weight='balanced'))
])

# Define hyperparameter grid
param_grid = {
    # TF-IDF params
    "tfidf__ngram_range": [(1, 1), (1, 2)],       # unigrams or unigrams+bigrams
    "tfidf__max_df": [0.85, 0.95, 1.0],           # ignore too-common words
    "tfidf__min_df": [1, 2, 5],                   # ignore too-rare words
    "tfidf__max_features": [3000, 5000, 10000],
    
    # Logistic Regression params
    "clf__C": [0.01, 0.1, 1, 5, 10],                    # inverse regularization strength
    "clf__penalty": ["l2"],                       # (l1 needs saga solver)
    "clf__solver": ["lbfgs"],
}

# Grid Search with 5-fold cross-validation
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=2
)
grid.fit(X_train, y_train)

# Best parameters and score
print("\nBest parameters found:")
print(grid.best_params_)
print(f"Best CV F1: {grid.best_score_:.3f}")

# Evaluate on test set
best_model = grid.best_estimator_
y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]

print("\nTest set results:")
print(classification_report(y_test, y_pred, digits=3))
print(f"AUC: {roc_auc_score(y_test, y_proba):.3f}")
